In [1]:
USER_WEIBO_DIR = r'..\data\crawler\weibo_crawler\weibo'

USER_SAVE_PATH = r'..\data\crawler\user_info.parquet'

WEIBO_SAVE_PATH = r'..\data\crawler\user_weibo.parquet'

In [2]:
import os

user_dirs = [d for d in os.listdir(USER_WEIBO_DIR)
             if os.path.isdir(os.path.join(USER_WEIBO_DIR, d))]

In [6]:
import os, json
import pandas as pd

user_info_rows = []   # df_user_info
weibo_rows        = []   # df_weibo

verified_map = {
    -1: "普通用户", 
    0: "个人认证", 
    1: "政府", 
    2: "企业",
    3: "媒体", 
    4: "校园", 
    5: "网站", 
    6: "应用",
    7: "团体/机构",
    200: "普通用户",  
    220: "个人认证"
}

for uid in user_dirs:
    json_path = os.path.join(USER_WEIBO_DIR, uid, f"{uid}.json")
    if not os.path.exists(json_path):
        continue
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    # ── 1. 用户基本信息 ───────────────────────────────────────
    user_info = data["user"]
    ip_location = user_info["ip_location"].split('（')[0]
    ip_location = ip_location if ip_location else "未知"
    registration_time = user_info["registration_time"]
    registration_time = registration_time if registration_time else "未知"
    verified_type = user_info["verified_type"]
    user_info_rows.append({
        "user_id": user_info["id"], 
        "screen_name": user_info["screen_name"], 
        "gender": user_info["gender"], 
        "ip_location": ip_location, 
        "registration_time": registration_time, 
        "total_weibo_count": user_info["statuses_count"], 
        "follower_count": user_info["followers_count"],
        "following_count": user_info["follow_count"], 
        "description": user_info["description"], 
        "verified": user_info["verified"], 
        "verified_type": verified_type, 
        "verified_type_name": verified_map.get(verified_type, "未知"),
        "user_rank": user_info["urank"]
    })

    # ── 2. 微博列表 ───────────────────────────────────────────
    for weibo in data["weibo"]:
        is_repost = True if "retweet" in weibo else False
        if is_repost:
            repost = weibo["retweet"]
            user_id = repost["user_id"]
            user_id = user_id if user_id else -1
            row_repost = {
                "weibo_id": repost["id"], 
                "user_id": user_id, 
                "screen_name": repost["screen_name"],
                "content": repost["text"], 
                "create_time": repost["created_at"], 
                # "ip_location": repost["location"] if repost["location"] else "未知", 
                "like_count": repost["attitudes_count"], 
                "comment_count": repost["comments_count"], 
                "repost_count": repost["reposts_count"],
                "topics": repost["topics"], 
                "at_users": repost["at_users"], 
                "reposted_weibo_id": -1
            }
        row = {
            "weibo_id": weibo["id"], 
            "user_id": weibo["user_id"], 
            "screen_name": weibo["screen_name"],
            "content": weibo["text"], 
            "create_time": weibo["created_at"], 
            # "ip_location": weibo["location"] if weibo["location"] else "未知", 
            "like_count": weibo["attitudes_count"], 
            "comment_count": weibo["comments_count"], 
            "repost_count": weibo["reposts_count"],
            "topics": weibo["topics"], 
            "at_users": weibo["at_users"], 
            "reposted_weibo_id": weibo["retweet"]["id"] if is_repost else -1
        }

        weibo_rows.append(row)
        weibo_rows.append(row_repost)

# ── 构建 DataFrame ────────────────────────────────────────────────────────
df_user_info = pd.DataFrame(user_info_rows)
df_weibo        = pd.DataFrame(weibo_rows)

In [7]:
df_user_info.to_parquet(USER_SAVE_PATH, index=False)
df_weibo.to_parquet(WEIBO_SAVE_PATH, index=False)